# Rebuild source-supported covariates (CDRv9 multiomics exports)



Builds a **reproducible, source-only** person-level covariate table from files in

`../resources/`:



1. `Multiomics_person-level_sample_counts_CDRv9 - lrWGS.phase2.ONT_v9.csv`

2. `Multiomics_person-level_sample_counts_CDRv9 - lrWGS.phase2_PacBio_v9.csv`

3. `Multiomics_person-level_sample_counts_CDRv9 - lrWGS.phase2.sample.final_set.2_Steve w v9 flag.csv`

4. `Multiomics_person-level_sample_counts_CDRv9 - multiomics_lists_CDRv9.csv`

5. `part.csv.gz` — demographics / EHR (`age`, `sex_at_birth`, `zip3_as_string`, `has_ehr_data`)

6. `lr_dna_extraction_method_cdrv9.tsv` — DNA extraction method

7. `trgt_table.txt` — TRGT VCF availability

8. `pc.log.gz` — tar archive of population-specific PCA files

9. `full.tsv.gz` — global full-training PCs

10. `lr.tsv` — headerless CDR v7/v8/v9 membership annotations

11. `AoU_Phase2_Phenotype.csv.gz` — phenotype-table membership (`in_phenotype_table`)
12. `counts.tar.gz` + `columns_structure.txt` — per-sample SV counts (`n_sv_*_ge50_*`)
13. `../resources/legacy_covariates/anc_df.csv.gz` — external ancestry predictions (`ancestry_pred`, `ancestry_pred_other`)
14. `../resources/legacy_covariates/aou_phase2.ped` — user-maintained Phase 2 pedigree annotations (no Phase 1 families)
15. `../resources/legacy_covariates/Integratedcall-GRCh38.tsv` — Phase 1 integrated-call coverage / status
16. `../resources/legacy_covariates/sv_sens_07_counts.json` — Phase 1 sensitivity SV totals (`n_sv_*_sens07`)
17. `../resources/legacy_covariates/ha_research_ids.csv` — HudsonAlpha biobank IDs
18. `../resources/legacy_covariates/lr_1027_sex_at_birth.csv` — Phase 1 sex-at-birth mapping
19. `ha_partial_sample.csv` + `../resources/legacy_covariates/ha_passing_sample.csv` — HA Chemagen/Autogen extraction platforms
20. `../resources/legacy_covariates/ont-sample-hg38.tsv` — curated Phase 1 ONT coverage / QC metrics
21. `../resources/legacy_covariates/merged_all_df.csv.gz` — conservative one-value-per-person gap fills



**Person universe:** all rows from the multiomics list (~17,226). Technical

lrWGS fields, demographics, and extraction method are left-joined where available.

Phase-1-only samples are not present in the CDRv9 ONT/PacBio technical sheets; for

those people `technology` / `platform` / `GC` / `techs_available` / `pb_meth_caller`

are filled as PacBio / Sequel / HA / PacBio / none from known Phase 1 metadata.

A curated set of Phase 1 research IDs also has additional ONT data; for those

people `techs_available` is set to `ONT|PacBio` and `has_ONT` is True

(primary `technology` stays PacBio; `has_ONT` may therefore mix v9-sheet and curated provenance).



**Construction rule:** never read `covariates.v3.csv.gz` while building the

output. That file is loaded only in the final comparison section.



All v3 fields are now represented from source data. Differences are retained and

reported rather than overwritten by v3.



**PCA provenance:** global full-training values from `full.tsv.gz` are written as

`PC1`–`PC32` (matching v3). The population-specific values from `pc.log.gz` are

retained as `pop_PC1`–`pop_PC32`; their `population` code comes from the archive

member name.



Writes:



- `../covariates.source_rebuilt.csv.gz`

- `../covariates.source_rebuilt.data_dictionary.tsv`



In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from __future__ import annotations

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py")
from workspace_paths import data_root

import json
import re
import tarfile

import numpy as np
import pandas as pd

ROOT = data_root()

RESOURCES = ROOT / "resources"
LEGACY_RESOURCES = RESOURCES / "legacy_covariates"

ONT_CSV = RESOURCES / "Multiomics_person-level_sample_counts_CDRv9 - lrWGS.phase2.ONT_v9.csv"
PACBIO_CSV = RESOURCES / "Multiomics_person-level_sample_counts_CDRv9 - lrWGS.phase2_PacBio_v9.csv"
FINAL_CSV = RESOURCES / (
    "Multiomics_person-level_sample_counts_CDRv9 - "
    "lrWGS.phase2.sample.final_set.2_Steve w v9 flag.csv"
)
MULTI_CSV = RESOURCES / "Multiomics_person-level_sample_counts_CDRv9 - multiomics_lists_CDRv9.csv"
PART_CSV = RESOURCES / "part.csv.gz"
EXTRACTION_TSV = RESOURCES / "lr_dna_extraction_method_cdrv9.tsv"
TRGT_TABLE = RESOURCES / "trgt_table.txt"
PCA_ARCHIVE = RESOURCES / "pc.log.gz"
FULL_PCA_TSV = RESOURCES / "full.tsv.gz"
CDR_MEMBERSHIP_TSV = RESOURCES / "lr.tsv"
PHENOTYPE_CSV = RESOURCES / "AoU_Phase2_Phenotype.csv.gz"
SV_COUNTS_TAR = RESOURCES / "counts.tar.gz"
SV_COLUMNS_STRUCTURE = RESOURCES / "columns_structure.txt"
ANCESTRY_CSV = LEGACY_RESOURCES / "anc_df.csv.gz"
PEDIGREE_CSV = LEGACY_RESOURCES / "aou_phase2.ped"
INTEGRATEDCALL_TSV = LEGACY_RESOURCES / "Integratedcall-GRCh38.tsv"
SV_SENS07_JSON = LEGACY_RESOURCES / "sv_sens_07_counts.json"
HA_RESEARCH_IDS_CSV = LEGACY_RESOURCES / "ha_research_ids.csv"
LR_1027_SEX_CSV = LEGACY_RESOURCES / "lr_1027_sex_at_birth.csv"
HA_PARTIAL_SAMPLE_CSV = ROOT / "ha_partial_sample.csv"
HA_PASSING_SAMPLE_CSV = LEGACY_RESOURCES / "ha_passing_sample.csv"
ONT_PHASE1_TSV = LEGACY_RESOURCES / "ont-sample-hg38.tsv"
MERGED_ALL_CSV = LEGACY_RESOURCES / "merged_all_df.csv.gz"

OUT_CSV = ROOT / "covariates.source_rebuilt.csv.gz"
OUT_DICT = ROOT / "covariates.source_rebuilt.data_dictionary.tsv"
V3_CSV = RESOURCES / "covariates.v3.csv.gz"
CONSTRUCTION_USED_V3 = False

TECH_COLS = [
    "participant",
    "long_read (1=final releasable in v9)",
    "biobank_id",
    "sex",
    "coverage",
    "platform",
    "PacBioMethylationCaller",
    "GC",
    "technology",
    "is_AIAN",
    "withdrew",
]
MULTI_COLS = [
    "research_id",
    "long_read (1=meet QC requirement)",
    "long_read phase",
    "long_read (1=final releasable in v9)",
    "RNASeq (1=meet QC requirement)",
    "RNASeq(1=final releasable)",
    "Proteomics (1=meet QC requirement)",
    "Proteomics(1=final releasable)",
    "Exposomics (1=releasable in v9)",
    "rid_no_srWGS",
]
PART_COLS = ["person_id", "age_at_cdr", "zip3_as_string", "sex_at_birth", "has_ehr_data"]
PC_COLS = [f"PC{i}" for i in range(1, 33)]
EXTRACTION_COLS = ["research_id", "extraction_method"]

print("ROOT:", ROOT)
print("RESOURCES:", RESOURCES)
for p in [
    ONT_CSV,
    PACBIO_CSV,
    FINAL_CSV,
    MULTI_CSV,
    PART_CSV,
    EXTRACTION_TSV,
    TRGT_TABLE,
    PCA_ARCHIVE,
    FULL_PCA_TSV,
    CDR_MEMBERSHIP_TSV,
    PHENOTYPE_CSV,
    SV_COUNTS_TAR,
    SV_COLUMNS_STRUCTURE,
    ANCESTRY_CSV,
    PEDIGREE_CSV,
    INTEGRATEDCALL_TSV,
    SV_SENS07_JSON,
    HA_RESEARCH_IDS_CSV,
    LR_1027_SEX_CSV,
    HA_PARTIAL_SAMPLE_CSV,
    HA_PASSING_SAMPLE_CSV,
    ONT_PHASE1_TSV,
    MERGED_ALL_CSV,
]:
    assert p.exists(), p
    print("  ok", p.name)


## 1. Load sources and validate schemas

IDs are read as strings. The multiomics list is the **primary key** for the
output (one row per `research_id`).


In [ ]:
def load_csv(path: Path, id_col: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert id_col in df.columns, f"{path.name}: missing {id_col}"
    df[id_col] = df[id_col].astype(str).str.strip()
    assert df[id_col].ne("").all(), f"{path.name}: blank {id_col}"
    return df


def require_columns(df: pd.DataFrame, cols: list[str], label: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    assert not missing, f"{label}: missing columns {missing}"


ont = load_csv(ONT_CSV, "participant")
pacbio = load_csv(PACBIO_CSV, "participant")
final = load_csv(FINAL_CSV, "participant")
multi = load_csv(MULTI_CSV, "research_id")
part = load_csv(PART_CSV, "person_id")

# User-provided Phase 1 metadata. These are source supplements, not inferred
# values: HA IDs supply biobank_id; the sex file fills only missing usable sex.
ha_biobank = pd.read_csv(
    HA_RESEARCH_IDS_CSV,
    usecols=["Research_ID", "Biobank_ID"],
    dtype=str,
    keep_default_na=False,
).rename(columns={"Research_ID": "research_id", "Biobank_ID": "ha_biobank_id"})
ha_biobank["research_id"] = ha_biobank["research_id"].astype(str).str.strip()
ha_biobank["ha_biobank_id"] = ha_biobank["ha_biobank_id"].replace("", pd.NA)
assert ha_biobank["research_id"].ne("").all()
assert ha_biobank["research_id"].is_unique, "ha_research_ids has duplicate Research_ID values"
assert ha_biobank["ha_biobank_id"].notna().all()

phase1_sex = pd.read_csv(
    LR_1027_SEX_CSV,
    dtype={"research_id": str, "sex_at_birth": str},
    keep_default_na=False,
).rename(columns={"sex_at_birth": "phase1_sex_at_birth"})
require_columns(phase1_sex, ["research_id", "phase1_sex_at_birth"], "lr_1027_sex_at_birth")
phase1_sex["research_id"] = phase1_sex["research_id"].astype(str).str.strip()
phase1_sex["phase1_sex_at_birth"] = phase1_sex["phase1_sex_at_birth"].replace({
    "": pd.NA,
    "No matching concept": pd.NA,
    "PMI: Skip": pd.NA,
})
assert phase1_sex["research_id"].ne("").all()
assert phase1_sex["research_id"].is_unique, "lr_1027_sex_at_birth has duplicate research_id values"
assert set(phase1_sex["phase1_sex_at_birth"].dropna()) == {"Female", "Male", "Intersex"}

extraction = pd.read_csv(EXTRACTION_TSV, sep="\t", dtype=str, keep_default_na=False)
extraction["research_id"] = extraction["research_id"].astype(str).str.strip()

# HudsonAlpha sample manifests carry Chemagen/Autogen labels keyed by biobank ID.
# Prefer the CDR extraction table; use these only to fill remaining gaps.
def load_ha_extraction(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    require_columns(raw, ["BIOBANK_ID", "Extraction Platform"], path.name)
    out = raw.rename(columns={
        "BIOBANK_ID": "biobank_id",
        "Extraction Platform": "ha_extraction_method",
    })[["biobank_id", "ha_extraction_method"]].copy()
    out["biobank_id"] = out["biobank_id"].astype(str).str.strip().replace({"": pd.NA})
    out["ha_extraction_method"] = out["ha_extraction_method"].replace({"": pd.NA})
    out = out.loc[out["biobank_id"].notna() & out["ha_extraction_method"].notna()].copy()
    out = out.drop_duplicates(subset=["biobank_id"], keep="first")
    assert set(out["ha_extraction_method"]).issubset({"Chemagen", "Autogen"})
    return out

ha_extraction = pd.concat(
    [
        load_ha_extraction(HA_PARTIAL_SAMPLE_CSV),
        load_ha_extraction(HA_PASSING_SAMPLE_CSV),
    ],
    ignore_index=True,
)
assert ha_extraction["biobank_id"].is_unique, "HA extraction manifests disagree on a biobank_id"
print(f"HA extraction platforms: {len(ha_extraction):,} biobank IDs")

# Curated Phase 1 ONT run metrics. These retain their own provenance and never
# replace the primary (PacBio-preferred) coverage field.
ont_phase1 = pd.read_csv(ONT_PHASE1_TSV, sep="\t", dtype=str, keep_default_na=False).rename(
    columns={
        "entity:ont-sample-hg38_id": "research_id",
        "mosdepth_cov": "ont_coverage",
        "aligned_num_bases": "ont_aligned_num_bases",
        "aligned_num_reads": "ont_aligned_num_reads",
        "aligned_read_length_N50": "ont_read_length_n50",
        "aligned_read_length_mean": "ont_read_length_mean",
        "aligned_read_length_median": "ont_read_length_median",
        "aligned_read_length_stdev": "ont_read_length_stdev",
        "average_identity": "ont_average_identity",
        "median_identity": "ont_median_identity",
    }
)
ONT_PHASE1_COLS = [
    "research_id", "ont_coverage", "ont_aligned_num_bases", "ont_aligned_num_reads",
    "ont_read_length_n50", "ont_read_length_mean", "ont_read_length_median",
    "ont_read_length_stdev", "ont_average_identity", "ont_median_identity",
]
require_columns(ont_phase1, ONT_PHASE1_COLS, ONT_PHASE1_TSV.name)
ont_phase1 = ont_phase1[ONT_PHASE1_COLS].copy()
ont_phase1["research_id"] = ont_phase1["research_id"].astype(str).str.strip()
assert ont_phase1["research_id"].ne("").all()
assert ont_phase1["research_id"].is_unique
for column in ONT_PHASE1_COLS[1:]:
    ont_phase1[column] = pd.to_numeric(ont_phase1[column].replace("", pd.NA), errors="coerce")
assert ont_phase1["ont_coverage"].notna().all()
print(f"Phase 1 ONT metrics: {len(ont_phase1):,} people")

# `merged_all_df` has repeated technical rows. It is retained raw here and is
# collapsed later only where a person has exactly one non-missing value.
merged_all = pd.read_csv(MERGED_ALL_CSV, dtype=str, keep_default_na=False)
require_columns(
    merged_all,
    ["person_id", "coverage", "sex_at_birth", "has_EHRs", "ancestry_pred", "ancestry_pred_other"],
    MERGED_ALL_CSV.name,
)
merged_all["person_id"] = merged_all["person_id"].astype(str).str.strip()
assert merged_all["person_id"].ne("").all()
print(f"merged_all records: {len(merged_all):,}; people: {merged_all['person_id'].nunique():,}")

# Keep only the interpretable ancestry labels. The supplied PC/probability columns
# are not joined: PC1–PC32 already have explicit full-training provenance here,
# and the probability-class ordering was not provided with this export.
ancestry = pd.read_csv(
    ANCESTRY_CSV,
    usecols=["person_id", "ancestry_pred", "ancestry_pred_other"],
    dtype=str,
    keep_default_na=False,
).rename(columns={"person_id": "research_id"})
ancestry["research_id"] = ancestry["research_id"].astype(str).str.strip()
assert ancestry["research_id"].ne("").all()
assert ancestry["research_id"].is_unique, "ancestry annotations have duplicate person IDs"
for column in ["ancestry_pred", "ancestry_pred_other"]:
    ancestry[column] = ancestry[column].replace("", pd.NA).str.lower()
ancestry["has_ancestry_annotation"] = True

# The pedigree marks known Phase 2 family structures (no Phase 1 families).
# Keep family-level fields useful for related-sample handling, but do not
# expose parental IDs in the person-level covariate export.
pedigree_raw = pd.read_csv(PEDIGREE_CSV, dtype=str, keep_default_na=False)
require_columns(pedigree_raw, ["ped", "id", "father", "mother", "sex", "affected", "avail"], "pedigree")
pedigree_raw["id"] = pedigree_raw["id"].astype(str).str.strip()
assert pedigree_raw["id"].ne("").all()
assert pedigree_raw["id"].is_unique, "pedigree has duplicate IDs"
pedigree = pd.DataFrame({
    "research_id": pedigree_raw["id"],
    "in_pedigree": True,
    "pedigree_family_id": pedigree_raw["ped"].replace("", pd.NA),
    "pedigree_is_founder": pedigree_raw["father"].eq("0") & pedigree_raw["mother"].eq("0"),
    "pedigree_n_parents": pedigree_raw[["father", "mother"]].ne("0").sum(axis=1).astype(int),
})
pedigree["pedigree_family_size"] = pedigree.groupby("pedigree_family_id")["research_id"].transform("size").astype(int)
assert pedigree["research_id"].is_unique

# The integrated-call export has 106 mostly file-path or redundant fields.
# Keep only Phase 1 coverage (to fill the shared coverage field) plus the
# historical approval/status annotation. Ignore identity / N50 / contamination.
INTEGRATEDCALL_ID = "entity:Integratedcall-GRCh38_id"
INTEGRATEDCALL_COLS = [
    INTEGRATEDCALL_ID,
    "GC_phaseI_status",
    "aligned_est_fold_cov",
]
integratedcall = pd.read_csv(
    INTEGRATEDCALL_TSV,
    sep="\t",
    usecols=INTEGRATEDCALL_COLS,
    dtype={INTEGRATEDCALL_ID: str, "GC_phaseI_status": str},
)
integratedcall = integratedcall.rename(columns={
    INTEGRATEDCALL_ID: "research_id",
    "GC_phaseI_status": "integratedcall_gc_phase1_status",
    "aligned_est_fold_cov": "integratedcall_aligned_fold_cov",
})
integratedcall["research_id"] = integratedcall["research_id"].astype(str).str.strip()
assert integratedcall["research_id"].ne("").all()
assert integratedcall["research_id"].is_unique, "Integratedcall metrics have duplicate IDs"
integratedcall["integratedcall_aligned_fold_cov"] = pd.to_numeric(
    integratedcall["integratedcall_aligned_fold_cov"], errors="coerce"
)
integratedcall["has_integratedcall_metrics"] = True
# trgt_table.txt has no header; multiple VCF paths for one person are expected.
trgt = pd.read_csv(
    TRGT_TABLE,
    sep="\t",
    header=None,
    names=["research_id", "trgt_vcf"],
    dtype=str,
    keep_default_na=False,
)
trgt["research_id"] = trgt["research_id"].astype(str).str.strip()

# Fabio's archive has four headerless center-specific CSVs. Columns 38–43
# (zero-based; documented in columns_structure.txt) are all INS/DEL ≥50 bp:
# DEL pav/pbsv/sniffles followed by INS pav/pbsv/sniffles.
SV_COUNT_COLUMNS = [
    "research_id",
    "sv_call_coverage",
    "n_sv_del_ge50_pav",
    "n_sv_del_ge50_pbsv",
    "n_sv_del_ge50_sniffles",
    "n_sv_ins_ge50_pav",
    "n_sv_ins_ge50_pbsv",
    "n_sv_ins_ge50_sniffles",
]
sv_frames = []
with tarfile.open(SV_COUNTS_TAR, "r:*") as archive:
    sv_members = [member for member in archive.getmembers() if member.isfile()]
    assert sv_members, f"No CSVs in {SV_COUNTS_TAR}"
    for member in sv_members:
        assert member.name.startswith("counts_") and member.name.endswith(".csv"), member.name
        with archive.extractfile(member) as fh:
            sv = pd.read_csv(fh, header=None, usecols=[0, 1, 38, 39, 40, 41, 42, 43])
        sv.columns = SV_COUNT_COLUMNS
        sv["research_id"] = sv["research_id"].astype(str).str.strip()
        sv["sv_source_gc"] = member.name.removeprefix("counts_").removesuffix(".csv").upper()
        sv_frames.append(sv)
sv_counts_raw = pd.concat(sv_frames, ignore_index=True)
assert sv_counts_raw["research_id"].ne("").all()
assert set(sv_counts_raw["sv_source_gc"]).issubset({"BI", "BCM", "HA", "UW"})

# Phase 1 sensitivity callset: INS/DEL/UNK totals (not caller-specific ≥50 bp).
with SV_SENS07_JSON.open() as fh:
    sv_sens07_raw = json.load(fh)
sv_sens07 = (
    pd.DataFrame.from_dict(sv_sens07_raw, orient="index")
    .rename_axis("research_id")
    .reset_index()
    .rename(columns={
        "DEL": "n_sv_del_sens07",
        "INS": "n_sv_ins_sens07",
    })
)
sv_sens07["research_id"] = sv_sens07["research_id"].astype(str).str.strip()
assert sv_sens07["research_id"].ne("").all()
assert sv_sens07["research_id"].is_unique
for column in ["n_sv_del_sens07", "n_sv_ins_sens07"]:
    assert column in sv_sens07.columns
    sv_sens07[column] = pd.to_numeric(sv_sens07[column], errors="raise").astype("Int64")
sv_sens07["has_sv_sens07_counts"] = True
print(f"sv_sens_07 counts: {len(sv_sens07):,} people")

# pc.log.gz is a tar archive whose member names encode population, e.g.
# pca/AoU_v9_AFR_training_pca.tsv. The PCs are population-specific.
pc_frames = []
with tarfile.open(PCA_ARCHIVE, "r:*") as archive:
    members = [
        member
        for member in archive.getmembers()
        if member.isfile() and member.name.endswith("_training_pca.tsv")
    ]
    assert members, f"No population PCA files in {PCA_ARCHIVE}"
    for member in members:
        match = re.search(r"AoU_v9_(.+)_training_pca\.tsv$", member.name)
        assert match, f"Unexpected PCA member name: {member.name}"
        population = match.group(1)
        with archive.extractfile(member) as fh:
            pc = pd.read_csv(fh, sep="\t", dtype={"s": str})
        expected_pc_cols = ["s", *PC_COLS]
        require_columns(pc, expected_pc_cols, member.name)
        pc = pc[expected_pc_cols].copy()
        pc["s"] = pc["s"].astype(str).str.strip()
        pc["population"] = population
        pc_frames.append(pc)

population_pca = pd.concat(pc_frames, ignore_index=True)
population_pca = population_pca.rename(columns={"s": "research_id", **{pc: f"pop_{pc}" for pc in PC_COLS}})
assert population_pca["research_id"].is_unique, "PCA archive assigns an ID to more than one population"

# Global PCs are the canonical PC1–PC32 covariates and match v3's PCA source.
global_pca = pd.read_csv(FULL_PCA_TSV, sep="\t", dtype={"s": str})
require_columns(global_pca, ["s", *PC_COLS], "full-training PCA")
global_pca = global_pca[["s", *PC_COLS]].rename(columns={"s": "research_id"})
global_pca["research_id"] = global_pca["research_id"].astype(str).str.strip()
assert global_pca["research_id"].is_unique, "full-training PCA has duplicate IDs"

# lr.tsv is a headerless, whitespace-delimited (research_id, CDR-version) list.
# One person can appear in multiple versions; aggregate membership with any().
cdr_membership = pd.read_csv(
    CDR_MEMBERSHIP_TSV,
    sep=r"\s+",
    header=None,
    names=["research_id", "cdr_version"],
    dtype=str,
)
cdr_membership["research_id"] = cdr_membership["research_id"].astype(str).str.strip()
cdr_membership["cdr_version"] = cdr_membership["cdr_version"].astype(str).str.strip().str.lower()
assert cdr_membership["research_id"].ne("").all()
assert cdr_membership["cdr_version"].isin({"v7", "v8", "v9"}).all()
cdr_flags = (
    cdr_membership.assign(present=True)
    .pivot_table(
        index="research_id",
        columns="cdr_version",
        values="present",
        aggfunc="any",
        fill_value=False,
    )
    .reindex(columns=["v7", "v8", "v9"], fill_value=False)
    .rename(columns={version: f"in_cdr_{version}" for version in ["v7", "v8", "v9"]})
    .reset_index()
)
assert cdr_flags["research_id"].is_unique

# Phenotype table is wide; only person_id is needed for membership.
phenotype_ids = pd.read_csv(PHENOTYPE_CSV, dtype={"person_id": str}, usecols=["person_id"])
phenotype_ids["person_id"] = phenotype_ids["person_id"].astype(str).str.strip()
assert phenotype_ids["person_id"].ne("").all()
assert phenotype_ids["person_id"].is_unique, "AoU_Phase2_Phenotype.csv.gz has duplicate person_id"
phenotype_id_set = set(phenotype_ids["person_id"])

require_columns(ont, TECH_COLS, "ONT")
require_columns(pacbio, TECH_COLS, "PacBio")
require_columns(final, TECH_COLS, "final")
require_columns(multi, MULTI_COLS, "multiomics")
require_columns(part, PART_COLS, "part")
require_columns(extraction, EXTRACTION_COLS, "extraction")

assert not CONSTRUCTION_USED_V3
assert multi["research_id"].is_unique
assert ont["participant"].is_unique
assert part["person_id"].is_unique
assert extraction["research_id"].is_unique
assert trgt["research_id"].ne("").all()
assert trgt["trgt_vcf"].ne("").all()
assert (ont["technology"] == "ONT").all()
assert (pacbio["technology"] == "PacBio").all()

summary = pd.DataFrame(
    [
        {"source": "multiomics_lists", "rows": len(multi), "unique_ids": multi["research_id"].nunique()},
        {"source": "part.csv.gz", "rows": len(part), "unique_ids": part["person_id"].nunique()},
        {"source": "extraction_method", "rows": len(extraction), "unique_ids": extraction["research_id"].nunique()},
        {"source": "TRGT table", "rows": len(trgt), "unique_ids": trgt["research_id"].nunique()},
        {"source": "SV counts ≥50 bp", "rows": len(sv_counts_raw), "unique_ids": sv_counts_raw["research_id"].nunique()},
        {"source": "population-specific PCA", "rows": len(population_pca), "unique_ids": population_pca["research_id"].nunique()},
        {"source": "global full-training PCA", "rows": len(global_pca), "unique_ids": global_pca["research_id"].nunique()},
        {"source": "CDR membership", "rows": len(cdr_membership), "unique_ids": cdr_flags["research_id"].nunique()},
        {"source": "phenotype table", "rows": len(phenotype_ids), "unique_ids": len(phenotype_id_set)},
        {"source": "ONT_v9", "rows": len(ont), "unique_ids": ont["participant"].nunique()},
        {"source": "PacBio_v9", "rows": len(pacbio), "unique_ids": pacbio["participant"].nunique()},
        {"source": "final_set_Steve", "rows": len(final), "unique_ids": final["participant"].nunique()},
    ]
)
display(summary)
print("schema checks passed; multiomics universe =", len(multi))


## 2. Collapse technical rows (for people with lrWGS)

Technical sequencing fields come from the Steve final-set for IDs present in the
ONT and/or PacBio v9 sheets. Duplicate participants are collapsed to **one
technical row**:

1. Prefer `technology == PacBio` over `ONT`
2. Then highest numeric `coverage`
3. Then lexicographic `platform`, `PacBioMethylationCaller`, `biobank_id`

Withdrawal / unavailability are **not** taken from this collapsed technical row.
They are derived from the full final-set in the next section (authoritative
consent/set-aside status).


In [ ]:
def parse_flag(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    true_vals = {"1", "TRUE", "True", "true", "YES", "Yes", "yes"}
    false_vals = {
        "0", "FALSE", "False", "false", "NO", "No", "no", "", "#N/A", "NA", "nan", "None", "<NA>"
    }
    out = pd.Series(pd.NA, index=series.index, dtype="boolean")
    out = out.mask(s.isin(true_vals), True)
    out = out.mask(s.isin(false_vals), False)
    unknown = ~s.isin(true_vals | false_vals)
    if unknown.any():
        print("WARNING: unmapped flag values:", sorted(s[unknown].unique())[:20])
    return out


def to_float(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().replace({"": np.nan, "#N/A": np.nan, "NA": np.nan})
    return pd.to_numeric(s, errors="coerce")


ont_ids = set(ont["participant"])
pacbio_ids = set(pacbio["participant"])
lr_universe = sorted(ont_ids | pacbio_ids)
print(
    f"lrWGS technical universe: {len(lr_universe):,} "
    f"(ONT-only={len(ont_ids - pacbio_ids):,}, "
    f"PacBio-only={len(pacbio_ids - ont_ids):,}, "
    f"both={len(ont_ids & pacbio_ids):,})"
)

final_u = final[final["participant"].isin(lr_universe)].copy()
missing_in_final = sorted(set(lr_universe) - set(final_u["participant"]))
assert not missing_in_final


def sheet_vs_final(sheet: pd.DataFrame, label: str) -> None:
    key = ["participant", "technology", "coverage", "platform"]
    hit = sheet.merge(final_u[key].drop_duplicates(), on=key, how="left", indicator=True)
    still = int((hit["_merge"] == "left_only").sum())
    print(f"{label} vs final on coverage+platform key: unmatched={still}")
    assert still == 0, f"{label} rows not found in final set"


sheet_vs_final(ont, "ONT")
sheet_vs_final(pacbio, "PacBio")

work = final_u.copy()
work["coverage_num"] = to_float(work["coverage"])
work["tech_rank"] = np.where(work["technology"] == "PacBio", 0, 1)
work["platform_key"] = work["platform"].fillna("")
work["meth_key"] = work["PacBioMethylationCaller"].fillna("")
work["biobank_key"] = work["biobank_id"].fillna("")
work = work.sort_values(
    by=["participant", "tech_rank", "coverage_num", "platform_key", "meth_key", "biobank_key"],
    ascending=[True, True, False, True, True, True],
    kind="mergesort",
)

audit = (
    work.groupby("participant", sort=False)
    .agg(
        n_lr_source_rows=("participant", "size"),
        technologies_available=("technology", lambda s: "|".join(sorted(set(s)))),
    )
    .reset_index()
)
chosen = work.groupby("participant", sort=False, as_index=False).head(1).copy()
assert chosen["participant"].is_unique

tech = chosen.merge(audit, on="participant", how="left", validate="one_to_one")
tech = tech.rename(
    columns={
        "participant": "research_id",
        "sex": "inferred_sex",
    }
)
tech = tech[
    [
        "research_id",
        "biobank_id",
        "GC",
        "technology",
        "platform",
        "PacBioMethylationCaller",
        "inferred_sex",
        "coverage",
        "is_AIAN",
        "n_lr_source_rows",
        "technologies_available",
    ]
].copy()
print("collapsed technical rows:", len(tech))
assert not CONSTRUCTION_USED_V3


## 3. Multiomics base + demographic, extraction, status, and technical left-joins

- Start from all multiomics rows (detailed QC / releasable flags).
- Left-join `part.csv.gz` → `age` (from `age_at_cdr`), `sex_at_birth`,
  `zip3_as_string`, `has_ehr_data`.
- Left-join `lr_dna_extraction_method_cdrv9.tsv` → `extraction_method`.
- After HA biobank-ID fill, fill remaining missing `extraction_method` from
  `ha_partial_sample.csv` / `ha_passing_sample.csv` (Chemagen/Autogen by biobank ID).
- Aggregate the headerless `trgt_table.txt` to person level → `has_trgt_calls` and
  `n_trgt_vcfs` (some people have multiple listed VCFs).
- Left-join global `PC1`–`PC32` from `full.tsv.gz` (the v3-compatible PCs).
- Left-join population-specific `pop_PC1`–`pop_PC32` and the population code from `pc.log.gz`.
- Aggregate the repeated ID records in `lr.tsv` into `in_cdr_v7`, `in_cdr_v8`, and
  `in_cdr_v9`, then left-join the flags.



- Derive `has_methylation` from `lr_phase`: True for `phase_2` and `phase_1_phase_2`; False for Phase-1-only and unlabeled.



- Derive `has_srWGS` from multiomics `rid_no_srWGS` (True unless that field marks



  the person as lacking short-read WGS); the raw echo-ID column is not retained.



- Mark `in_phenotype_table` from presence of `research_id` in



  `AoU_Phase2_Phenotype.csv.gz` (`person_id`); does not embed phenotype values.



- From the **full** Steve final-set (not just the ONT/PacBio releasable subset),



  derive person-level status:



  - `withdrawn`: any final-set row has `withdrew=TRUE` (authoritative)



  - `unavailable`: in the final-set, not withdrawn, and no row is v9-releasable



    (set-aside / otherwise held out; release flag `#N/A`)



- Left-join collapsed lrWGS technical fields for ONT∪PacBio people.







- Public column names are short/interpretable; redundant raw 1/NA multiomics echoes are dropped in favor of bools.





In [ ]:
base = multi.rename(
    columns={
        "long_read (1=meet QC requirement)": "long_read_meet_qc",
        "long_read phase": "long_read_phase",
        "long_read (1=final releasable in v9)": "long_read_releasable_v9",
        "RNASeq (1=meet QC requirement)": "rnaseq_meet_qc",
        "RNASeq(1=final releasable)": "rnaseq_releasable",
        "Proteomics (1=meet QC requirement)": "proteomics_meet_qc",
        "Proteomics(1=final releasable)": "proteomics_releasable",
        "Exposomics (1=releasable in v9)": "exposomics_releasable_v9",
    }
).copy()

part_join = part[PART_COLS].rename(
    columns={"person_id": "research_id", "age_at_cdr": "age"}
).copy()
part_join["age"] = to_float(part_join["age"])
part_join["zip3_as_string"] = part_join["zip3_as_string"].replace({"": pd.NA})
part_join["sex_at_birth"] = part_join["sex_at_birth"].replace({"": pd.NA})
part_join["has_ehr_data"] = parse_flag(part_join["has_ehr_data"])  # nullable bool; unmatched stay NA

n_before = len(base)
built = base.merge(part_join, on="research_id", how="left", validate="one_to_one", indicator="_part_merge")
assert len(built) == n_before
print("part.csv.gz left-join:")
print(built["_part_merge"].value_counts())
built["has_demographics"] = built["_part_merge"].eq("both")
built = built.drop(columns=["_part_merge"])

# Preserve usable demographics from part.csv.gz; use the supplied Phase 1 file
# only for gaps. “No matching concept” and “PMI: Skip” were normalized to NA
# at load time and therefore cannot overwrite a usable sex value.
built = built.merge(phase1_sex, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
phase1_related = built["long_read_phase"].isin(["phase_1", "phase_1_phase_2"])
assert built.loc[phase1_related, "phase1_sex_at_birth"].notna().sum() == 1007
sex_fill = built["sex_at_birth"].isna() & built["phase1_sex_at_birth"].notna()
built.loc[sex_fill, "sex_at_birth"] = built.loc[sex_fill, "phase1_sex_at_birth"]
built = built.drop(columns=["phase1_sex_at_birth"])
print("Phase 1 sex-at-birth fills from lr_1027:", int(sex_fill.sum()))

extraction_join = extraction[EXTRACTION_COLS].copy()
built = built.merge(
    extraction_join,
    on="research_id",
    how="left",
    validate="one_to_one",
    indicator="_extraction_merge",
)
assert len(built) == n_before
print("extraction-method left-join:")
print(built["_extraction_merge"].value_counts())
built["has_extraction_method"] = built["_extraction_merge"].eq("both")
built = built.drop(columns=["_extraction_merge"])

trgt_summary = (
    trgt.groupby("research_id", as_index=False)
    .agg(n_trgt_vcfs=("trgt_vcf", "size"))
    .assign(has_trgt_calls=True)
)
assert trgt_summary["research_id"].is_unique
built = built.merge(trgt_summary, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
built["has_trgt_calls"] = built["has_trgt_calls"].fillna(False).astype(bool)
built["n_trgt_vcfs"] = built["n_trgt_vcfs"].fillna(0).astype(int)
assert (built.loc[built["has_trgt_calls"], "n_trgt_vcfs"] > 0).all()
assert (built.loc[~built["has_trgt_calls"], "n_trgt_vcfs"] == 0).all()
print("TRGT availability:")
print(built["has_trgt_calls"].value_counts().to_dict())
print("people with multiple TRGT VCFs:", int((built["n_trgt_vcfs"] > 1).sum()))

built = built.merge(
    global_pca,
    on="research_id",
    how="left",
    validate="one_to_one",
    indicator="_global_pca_merge",
)
assert len(built) == n_before
print("global full-training PCA left-join:")
print(built["_global_pca_merge"].value_counts())
built["has_global_pcs"] = built["_global_pca_merge"].eq("both")
built = built.drop(columns=["_global_pca_merge"])

built = built.merge(
    population_pca,
    on="research_id",
    how="left",
    validate="one_to_one",
    indicator="_population_pca_merge",
)
assert len(built) == n_before
print("population-specific PCA left-join:")
print(built["_population_pca_merge"].value_counts())
built["has_population_pcs"] = built["_population_pca_merge"].eq("both")
built = built.drop(columns=["_population_pca_merge"])
assert built.loc[built["has_population_pcs"], "population"].notna().all()
assert built.loc[~built["has_population_pcs"], "population"].isna().all()

built = built.merge(ancestry, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
built["has_ancestry_annotation"] = built["has_ancestry_annotation"].fillna(False).astype(bool)
print("external ancestry annotation availability:", built["has_ancestry_annotation"].value_counts().to_dict())
print("ancestry_pred_other among annotated:")
print(built.loc[built["has_ancestry_annotation"], "ancestry_pred_other"].value_counts().to_dict())

built = built.merge(pedigree, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
built["in_pedigree"] = built["in_pedigree"].fillna(False).astype(bool)
for column in ["pedigree_is_founder"]:
    built[column] = built[column].fillna(False).astype(bool)
for column in ["pedigree_n_parents", "pedigree_family_size"]:
    built[column] = built[column].astype("Int64")
print("pedigree availability:", built["in_pedigree"].value_counts().to_dict())
print("pedigree families represented:", int(built.loc[built["in_pedigree"], "pedigree_family_id"].nunique()))
assert not built.loc[built["long_read_phase"].eq("phase_1"), "in_pedigree"].any(), (
    "Phase 1 people unexpectedly present in Phase 2 pedigree file"
)

built = built.merge(cdr_flags, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
for column in ["in_cdr_v7", "in_cdr_v8", "in_cdr_v9"]:
    built[column] = built[column].fillna(False).astype(bool)
print("CDR membership counts:")
print(built[["in_cdr_v7", "in_cdr_v8", "in_cdr_v9"]].sum().to_dict())

# Methylation was produced for Phase 2 sequencing. Phase-1-only samples lack it;
# Phase 1 samples re-sequenced in Phase 2 (`phase_1_phase_2`) have it.
built["has_methylation"] = built["long_read_phase"].isin(["phase_2", "phase_1_phase_2"])
print("has_methylation:", built["has_methylation"].value_counts().to_dict())
print(
    "has_methylation by long_read_phase:",
    built.groupby(built["long_read_phase"].fillna("NA"))["has_methylation"]
    .sum()
    .astype(int)
    .to_dict(),
)
assert not built.loc[built["long_read_phase"].eq("phase_1"), "has_methylation"].any()
assert built.loc[
    built["long_read_phase"].isin(["phase_2", "phase_1_phase_2"]), "has_methylation"
].all()

built["in_phenotype_table"] = built["research_id"].isin(phenotype_id_set)
print("in_phenotype_table:", built["in_phenotype_table"].value_counts().to_dict())

# Authoritative person-level long-read status from the full final-set.
final_status = final.copy()
final_status["participant"] = final_status["participant"].astype(str).str.strip()
final_status["_withdrew"] = parse_flag(final_status["withdrew"]).fillna(False)
final_status["_releasable"] = parse_flag(
    final_status["long_read (1=final releasable in v9)"]
).fillna(False)
final_status = (
    final_status.groupby("participant", as_index=False)
    .agg(
        withdrawn=("_withdrew", "any"),
        final_set_releasable_v9=("_releasable", "any"),
        n_final_set_rows=("participant", "size"),
    )
    .rename(columns={"participant": "research_id"})
)
final_status["unavailable"] = (
    ~final_status["withdrawn"] & ~final_status["final_set_releasable_v9"]
)
assert not (
    final_status["withdrawn"] & final_status["unavailable"]
).any(), "withdrawn and unavailable must be mutually exclusive"

built = built.merge(final_status, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
built["in_final_set"] = built["n_final_set_rows"].notna()
built["withdrawn"] = built["withdrawn"].fillna(False).astype(bool)
built["unavailable"] = built["unavailable"].fillna(False).astype(bool)
built["final_set_releasable_v9"] = built["final_set_releasable_v9"].fillna(False).astype(bool)
built["n_final_set_rows"] = built["n_final_set_rows"].fillna(0).astype(int)
print("long-read status on multiomics universe:")
print(
    {
        "withdrawn": int(built["withdrawn"].sum()),
        "unavailable": int(built["unavailable"].sum()),
        "final_set_releasable_v9": int(built["final_set_releasable_v9"].sum()),
        "in_final_set": int(built["in_final_set"].sum()),
    }
)
assert int(built["withdrawn"].sum()) == 123
assert int(built["unavailable"].sum()) == 9

built = built.merge(tech, on="research_id", how="left", validate="one_to_one", indicator="_tech_merge")
assert len(built) == n_before
print("technical left-join:")
print(built["_tech_merge"].value_counts())
built["has_lr_technical"] = built["_tech_merge"].eq("both")
built = built.drop(columns=["_tech_merge"])

# Phase 1 was PacBio Sequel at HudsonAlpha and is absent from the CDRv9
# ONT/PacBio technical sheets used for the collapse above. Fill known Phase 1
# instrument / center / tech-availability fields; methylation was not produced.
phase1_missing_tech = built["long_read_phase"].eq("phase_1") & built["technology"].isna()
assert phase1_missing_tech.equals(
    built["long_read_phase"].eq("phase_1") & ~built["has_lr_technical"]
), "unexpected Phase 1 rows with partial technical metadata"
built.loc[phase1_missing_tech, "technology"] = "PacBio"
built.loc[phase1_missing_tech, "platform"] = "Sequel"
built.loc[phase1_missing_tech, "GC"] = "HA"
built.loc[phase1_missing_tech, "technologies_available"] = "PacBio"
built.loc[phase1_missing_tech, "PacBioMethylationCaller"] = "none"
# Phase 1 has one known PacBio run even though it is absent from v9 sheets.
built.loc[phase1_missing_tech, "n_lr_source_rows"] = 1
built["n_lr_source_rows"] = pd.to_numeric(built["n_lr_source_rows"], errors="coerce").astype("Int64")
print(
    "Phase 1 PacBio/Sequel/HA fill:",
    int(phase1_missing_tech.sum()),
    "people (has_lr_technical remains False; coverage from integratedcall)",
)

# Curated Phase 1 samples with additional ONT data (not in CDRv9 ONT_v9 sheet).
# Keep primary technology=PacBio; annotate dual-tech availability only.
PHASE1_ADDITIONAL_ONT_IDS = frozenset({
    "1000513", "1000920", "1001399", "1001980", "1002322", "1002826", "1004266",
    "1005038", "1005444", "1005938", "1007198", "1008775", "1010384", "1012440",
    "1012736", "1013536", "1014457", "1014625", "1014694", "1014764", "1014823",
    "1015059", "1015507", "1016971", "1016985", "1019345", "1024761", "1025136",
    "1025342", "1025566", "1025694", "1026351", "1026529", "1026622", "1027488",
    "1027673", "1029520", "1029873", "1032052", "1032684", "1036042", "1037292",
    "1037774", "1037790", "1037792", "1037950", "1041753", "1042609", "1044452",
    "1048940",
})
phase1_ont = built["research_id"].isin(PHASE1_ADDITIONAL_ONT_IDS)
assert int(phase1_ont.sum()) == len(PHASE1_ADDITIONAL_ONT_IDS)
assert built.loc[phase1_ont, "long_read_phase"].isin(["phase_1", "phase_1_phase_2"]).all()
assert built.loc[phase1_ont, "technology"].eq("PacBio").all()

def _add_ont_tech(value: object) -> str:
    parts = {p for p in str(value).split("|") if p and p != "nan"} if pd.notna(value) else set()
    parts.add("ONT")
    parts.add("PacBio")
    return "|".join(sorted(parts))

built.loc[phase1_ont, "technologies_available"] = built.loc[phase1_ont, "technologies_available"].map(_add_ont_tech)
assert built.loc[phase1_ont, "technologies_available"].eq("ONT|PacBio").all()
print(
    "Phase 1 additional ONT annotation:",
    int(phase1_ont.sum()),
    "people (techs_available=ONT|PacBio; has_ONT=True including curated list)",
)

built = built.merge(ont_phase1, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
assert built.loc[phase1_ont, "ont_coverage"].notna().all()
assert built.loc[~phase1_ont, "ont_coverage"].isna().all()
print("Phase 1 ONT coverage/QC metrics joined:", int(built["ont_coverage"].notna().sum()))

# The HA mapping provides participant-level biobank IDs. Technical final-set
# values stay authoritative where present; fill Phase-1-only values that are
# absent from the v9 technical sheets.
built = built.merge(ha_biobank, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
assert built.loc[phase1_missing_tech, "ha_biobank_id"].notna().all()
biobank_fill = built["biobank_id"].isna() & built["ha_biobank_id"].notna()
built.loc[biobank_fill, "biobank_id"] = built.loc[biobank_fill, "ha_biobank_id"]
built = built.drop(columns=["ha_biobank_id"])
assert built.loc[phase1_missing_tech, "biobank_id"].notna().all()
print("HA biobank-ID fills:", int(biobank_fill.sum()))

# Prefer CDR extraction_method; fill remaining gaps from HA sample manifests.
n_missing_before_ha_ext = int(built["extraction_method"].isna().sum())
built = built.merge(ha_extraction, on="biobank_id", how="left", validate="many_to_one")
assert len(built) == n_before
ha_ext_fill = built["extraction_method"].isna() & built["ha_extraction_method"].notna()
built.loc[ha_ext_fill, "extraction_method"] = built.loc[ha_ext_fill, "ha_extraction_method"]
built = built.drop(columns=["ha_extraction_method"])
built["has_extraction_method"] = built["extraction_method"].notna()
print(
    "HA extraction fills:",
    int(ha_ext_fill.sum()),
    f"(missing before={n_missing_before_ha_ext:,}; after={int(built['extraction_method'].isna().sum()):,})",
)
assert built.loc[phase1_missing_tech, "extraction_method"].notna().all()
assert built.loc[built["long_read_phase"].eq("phase_1"), "extraction_method"].notna().all(), (
    "Phase 1 extraction_method still incomplete after HA fill"
)

built = built.merge(integratedcall, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
built["has_integratedcall_metrics"] = built["has_integratedcall_metrics"].fillna(False).astype(bool)
# These metrics describe the historical Phase 1 Sequel run. They cover every
# phase_1-only person and the 14 people subsequently re-sequenced in Phase 2.
phase1_integrated = built["long_read_phase"].isin(["phase_1", "phase_1_phase_2"])
assert built.loc[phase1_integrated, "has_integratedcall_metrics"].all()
assert not built.loc[~phase1_integrated, "has_integratedcall_metrics"].any()
# Fill the shared coverage field only where Phase 1 lacks a v9 technical row;
# retain Phase 2's final-set coverage for people with repeat sequencing.
built["coverage"] = to_float(built["coverage"])
phase1_coverage_fill = built["long_read_phase"].eq("phase_1") & built["coverage"].isna()
assert phase1_coverage_fill.equals(built["long_read_phase"].eq("phase_1"))
built.loc[phase1_coverage_fill, "coverage"] = built.loc[
    phase1_coverage_fill, "integratedcall_aligned_fold_cov"
]
built["coverage_source"] = pd.NA
built.loc[built["coverage"].notna() & built["has_lr_technical"], "coverage_source"] = "v9_technical"
built.loc[phase1_coverage_fill, "coverage_source"] = "integratedcall_phase1"
assert built.loc[phase1_coverage_fill, "coverage"].notna().all()
print(
    "Integratedcall metrics:",
    int(built["has_integratedcall_metrics"].sum()),
    "; Phase 1 coverage filled:",
    int(phase1_coverage_fill.sum()),
)

# `merged_all_df` supplements only gaps: collapse each field only when a person
# has one non-missing normalized value across all of their source rows.
merged_fill = merged_all[[
    "person_id", "coverage", "sex_at_birth", "has_EHRs", "ancestry_pred", "ancestry_pred_other",
]].rename(columns={"person_id": "research_id"}).copy()
merged_fill = merged_fill.replace("", pd.NA)
merged_fill["merged_coverage"] = pd.to_numeric(merged_fill.pop("coverage"), errors="coerce")
merged_fill["merged_sex_at_birth"] = merged_fill.pop("sex_at_birth").replace({
    "PMI: Skip": pd.NA,
    "I prefer not to answer": pd.NA,
    "Sex At Birth: Sex At Birth None Of These": pd.NA,
})
assert set(merged_fill["merged_sex_at_birth"].dropna()).issubset({"Female", "Male", "Intersex"})
merged_fill["merged_has_ehr_data"] = merged_fill.pop("has_EHRs").map({"True": True, "False": False})
assert merged_fill["merged_has_ehr_data"].notna().sum() == merged_all["has_EHRs"].replace("", pd.NA).notna().sum()
merged_fill["merged_ancestry_pred"] = merged_fill.pop("ancestry_pred").str.lower()
merged_fill["merged_ancestry_pred_other"] = merged_fill.pop("ancestry_pred_other").str.lower()
MERGED_FILL_COLS = [
    "merged_coverage", "merged_sex_at_birth", "merged_has_ehr_data",
    "merged_ancestry_pred", "merged_ancestry_pred_other",
]

def unique_nonmissing(series: pd.Series) -> object:
    values = pd.unique(series.dropna())
    return values[0] if len(values) == 1 else pd.NA

merged_fill = merged_fill.groupby("research_id", as_index=False)[MERGED_FILL_COLS].agg(unique_nonmissing)
assert merged_fill["research_id"].is_unique
built = built.merge(merged_fill, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before

merged_coverage_fill = built["coverage"].isna() & built["merged_coverage"].notna()
built.loc[merged_coverage_fill, "coverage"] = built.loc[
    merged_coverage_fill, "merged_coverage"
].astype(float).to_numpy()
built.loc[merged_coverage_fill, "coverage_source"] = "merged_all_df"
merged_sex_fill = built["sex_at_birth"].isna() & built["merged_sex_at_birth"].notna()
built.loc[merged_sex_fill, "sex_at_birth"] = built.loc[merged_sex_fill, "merged_sex_at_birth"]
merged_ehr_fill = built["has_ehr_data"].isna() & built["merged_has_ehr_data"].notna()
built.loc[merged_ehr_fill, "has_ehr_data"] = built.loc[merged_ehr_fill, "merged_has_ehr_data"]
merged_ancestry_fill = built["ancestry_pred"].isna() & built["merged_ancestry_pred"].notna()
built.loc[merged_ancestry_fill, "ancestry_pred"] = built.loc[merged_ancestry_fill, "merged_ancestry_pred"]
merged_ancestry_other_fill = built["ancestry_pred_other"].isna() & built["merged_ancestry_pred_other"].notna()
built.loc[merged_ancestry_other_fill, "ancestry_pred_other"] = built.loc[
    merged_ancestry_other_fill, "merged_ancestry_pred_other"
]
built["has_ancestry_annotation"] = built["ancestry_pred"].notna() | built["ancestry_pred_other"].notna()
assert built.loc[merged_coverage_fill, "coverage_source"].eq("merged_all_df").all()
print(
    "merged_all conservative fills:",
    {
        "coverage": int(merged_coverage_fill.sum()),
        "sex_at_birth": int(merged_sex_fill.sum()),
        "has_ehr_data": int(merged_ehr_fill.sum()),
        "ancestry_pred": int(merged_ancestry_fill.sum()),
        "ancestry_pred_other": int(merged_ancestry_other_fill.sum()),
    },
)
built = built.drop(columns=MERGED_FILL_COLS)

# Resolve 28 repeated SV-count IDs by the source-count file that matches the
# collapsed sequencing-row genome center. Non-repeated IDs need no resolution.
sv_lookup = sv_counts_raw.merge(
    built[["research_id", "GC"]], on="research_id", how="inner", validate="many_to_one"
)
sv_row_count = sv_lookup.groupby("research_id")["research_id"].transform("size")
sv_counts = sv_lookup.loc[
    sv_row_count.eq(1) | sv_lookup["sv_source_gc"].eq(sv_lookup["GC"])
].copy()
assert sv_counts["research_id"].is_unique
assert set(sv_counts["research_id"]) == (
    set(sv_counts_raw["research_id"]) & set(built["research_id"])
)
assert (
    sv_counts.loc[sv_counts["research_id"].isin(
        sv_counts_raw.loc[sv_counts_raw["research_id"].duplicated(keep=False), "research_id"]
    ), "sv_source_gc"]
    == sv_counts.loc[sv_counts["research_id"].isin(
        sv_counts_raw.loc[sv_counts_raw["research_id"].duplicated(keep=False), "research_id"]
    ), "GC"]
).all()
sv_counts = sv_counts[SV_COUNT_COLUMNS]

built = built.merge(sv_counts, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
built["has_sv_counts"] = built["sv_call_coverage"].notna()
for column in SV_COUNT_COLUMNS[2:]:
    built[column] = built[column].astype("Int64")
print("SV-count availability (counts.tar.gz):", built["has_sv_counts"].value_counts().to_dict())
assert int(built["has_sv_counts"].sum()) == 12391

# Phase 1 sensitivity SV totals (INS/DEL/UNK). Covers every phase_1 person and
# the 14 phase_1_phase_2 people; caller-specific ≥50 bp counts remain from the
# tar for Phase 2 / re-sequenced samples.
built = built.merge(sv_sens07, on="research_id", how="left", validate="one_to_one")
assert len(built) == n_before
built["has_sv_sens07_counts"] = built["has_sv_sens07_counts"].fillna(False).astype(bool)
for column in ["n_sv_del_sens07", "n_sv_ins_sens07"]:
    built[column] = built[column].astype("Int64")
phase1_sv = built["long_read_phase"].eq("phase_1")
assert built.loc[phase1_sv, "has_sv_sens07_counts"].all(), "Phase 1 missing sv_sens_07 counts"
assert built.loc[
    built["long_read_phase"].isin(["phase_1", "phase_1_phase_2"]), "has_sv_sens07_counts"
].all()
print(
    "sv_sens_07 availability:",
    built["has_sv_sens07_counts"].value_counts().to_dict(),
    "; Phase 1 with sens07:",
    int(built.loc[phase1_sv, "has_sv_sens07_counts"].sum()),
)

# Publish one deletion and one insertion count. Phase 2 uses the maximum
# caller-specific ≥50 bp count; Phase 1 uses its sens07 total. For the 14
# people sequenced in both phases, the Phase 2 caller maximum takes precedence.
phase2_del = built[[
    "n_sv_del_ge50_pav", "n_sv_del_ge50_pbsv", "n_sv_del_ge50_sniffles"
]].max(axis=1, skipna=True)
phase2_ins = built[[
    "n_sv_ins_ge50_pav", "n_sv_ins_ge50_pbsv", "n_sv_ins_ge50_sniffles"
]].max(axis=1, skipna=True)
built["n_sv_del"] = phase2_del.combine_first(built["n_sv_del_sens07"]).astype("Int64")
built["n_sv_ins"] = phase2_ins.combine_first(built["n_sv_ins_sens07"]).astype("Int64")
assert built.loc[phase1_sv, ["n_sv_del", "n_sv_ins"]].notna().all().all()
assert built.loc[built["long_read_phase"].eq("phase_2") & built["has_sv_counts"], ["n_sv_del", "n_sv_ins"]].notna().all().all()
print("unified SV count availability:", {
    "n_sv_del": int(built["n_sv_del"].notna().sum()),
    "n_sv_ins": int(built["n_sv_ins"].notna().sum()),
})

built["has_ONT"] = built["research_id"].isin(ont_ids) | built["research_id"].isin(PHASE1_ADDITIONAL_ONT_IDS)
assert built.loc[built["research_id"].isin(PHASE1_ADDITIONAL_ONT_IDS), "has_ONT"].all()
built["has_PacBio"] = built["research_id"].isin(pacbio_ids)
built["rid_no_srWGS"] = built["rid_no_srWGS"].replace({"#N/A": pd.NA, "": pd.NA})
built["has_srWGS"] = built["rid_no_srWGS"].isna()
print("has_srWGS:", built["has_srWGS"].value_counts().to_dict())
assert int((~built["has_srWGS"]).sum()) == int(built["rid_no_srWGS"].notna().sum())

for raw, outcol in {
    "long_read_meet_qc": "long_read_meet_qc_bool",
    "long_read_releasable_v9": "long_read_releasable_v9_bool",
    "rnaseq_meet_qc": "rnaseq_meet_qc_bool",
    "rnaseq_releasable": "has_rna",
    "proteomics_meet_qc": "proteomics_meet_qc_bool",
    "proteomics_releasable": "has_proteomics",
    "exposomics_releasable_v9": "has_exposomics",
}.items():
    built[outcol] = parse_flag(built[raw])

built["is_AIAN"] = parse_flag(built["is_AIAN"])
built["coverage"] = to_float(built["coverage"])
built["PacBioMethylationCaller"] = built["PacBioMethylationCaller"].replace("", pd.NA)
built["long_read_phase"] = built["long_read_phase"].replace("", pd.NA)

for col in [
    "long_read_meet_qc",
    "long_read_releasable_v9",
    "rnaseq_meet_qc",
    "rnaseq_releasable",
    "proteomics_meet_qc",
    "proteomics_releasable",
    "exposomics_releasable_v9",
]:
    built[col] = built[col].replace({"": pd.NA, "#N/A": pd.NA})
    built[col] = pd.to_numeric(built[col], errors="coerce").astype("Int64")

print(f"rows={len(built):,}")
print("has_demographics:", built["has_demographics"].value_counts().to_dict())
print("has_extraction_method:", built["has_extraction_method"].value_counts().to_dict())
print("has_lr_technical:", built["has_lr_technical"].value_counts().to_dict())
print("age missing:", int(built["age"].isna().sum()))
assert not CONSTRUCTION_USED_V3


## 4. Assemble output table and write artifacts


In [ ]:
out_cols = [



    "research_id",



    "lr_phase",



    "lr_meet_qc",



    "lr_releasable_v9",



    "rna_meet_qc",



    "has_rna",



    "proteomics_meet_qc",



    "has_proteomics",



    "has_exposomics",



    "has_srWGS",



    "age",



    "sex_at_birth",



    "zip3",



    "has_ehr_data",



    "has_demographics",



    "extraction_method",



    "has_extraction_method",



    "has_trgt",



    "n_trgt_vcfs",



    "population",
    "ancestry_pred",
    "ancestry_pred_other",
    "has_ancestry_annotation",
    "in_pedigree",
    "pedigree_family_id",
    "pedigree_is_founder",
    "pedigree_n_parents",
    "pedigree_family_size",



    *PC_COLS,



    *[f"pop_PC{i}" for i in range(1, 33)],



    "has_global_pcs",



    "has_pop_pcs",



    "in_cdr_v7",



    "in_cdr_v8",



    "in_cdr_v9",



    "has_methylation",



    "in_phenotype_table",



    "biobank_id",



    "GC",



    "technology",



    "platform",



    "pb_meth_caller",



    "inferred_sex",



    "coverage",
    "coverage_source",
    "ont_coverage",
    "ont_aligned_num_bases",
    "ont_aligned_num_reads",
    "ont_read_length_n50",
    "ont_read_length_mean",
    "ont_read_length_median",
    "ont_read_length_stdev",
    "ont_average_identity",
    "ont_median_identity",
    "has_sv_counts",
    "n_sv_del",
    "n_sv_ins",



    "is_AIAN",



    "withdrawn",



    "unavailable",



    "final_releasable_v9",



    "in_final_set",



    "n_final_rows",



    "has_lr_tech",



    "has_ONT",



    "has_PacBio",



    "n_lr_technical_rows",



    "techs_available",



]







# Public names: short/interpretable; drop redundant raw 1/NA multiomics echoes.



rename_to_public = {



    "long_read_phase": "lr_phase",



    "long_read_meet_qc_bool": "lr_meet_qc",



    "long_read_releasable_v9_bool": "lr_releasable_v9",



    "rnaseq_meet_qc_bool": "rna_meet_qc",



    "proteomics_meet_qc_bool": "proteomics_meet_qc",



    "zip3_as_string": "zip3",



    "has_trgt_calls": "has_trgt",



    "has_population_pcs": "has_pop_pcs",



    "PacBioMethylationCaller": "pb_meth_caller",



    "final_set_releasable_v9": "final_releasable_v9",



    "n_final_set_rows": "n_final_rows",



    "has_lr_technical": "has_lr_tech",









    "n_lr_source_rows": "n_lr_technical_rows",



    "technologies_available": "techs_available",



}







# Drop raw 1/NA echoes before rename to avoid name collisions (e.g. proteomics_meet_qc).

drop_raw = [

    "long_read_meet_qc",

    "long_read_releasable_v9",

    "rnaseq_meet_qc",

    "rnaseq_releasable",

    "proteomics_meet_qc",

    "proteomics_releasable",

    "exposomics_releasable_v9",

]

public = built.drop(columns=[c for c in drop_raw if c in built.columns]).rename(columns=rename_to_public)



missing = [c for c in out_cols if c not in public.columns]



assert not missing, f"missing public columns: {missing}"



out = public[out_cols].copy()



out = out.sort_values("research_id", kind="mergesort").reset_index(drop=True)



assert out["research_id"].is_unique



assert len(out) == len(multi)



assert not CONSTRUCTION_USED_V3







data_dictionary = pd.DataFrame(



    [



        {"column": "research_id", "source": "multiomics research_id", "notes": "primary key"},



        {"column": "lr_phase", "source": "multiomics long_read phase", "notes": "phase_1 / phase_2 / phase_1_phase_2"},



        {"column": "lr_meet_qc", "source": "multiomics long_read meet QC", "notes": "bool; blank→False"},



        {"column": "lr_releasable_v9", "source": "multiomics long_read releasable v9", "notes": "bool; blank→False"},



        {"column": "rna_meet_qc", "source": "multiomics RNASeq meet QC", "notes": "bool; blank→False"},



        {"column": "has_rna", "source": "multiomics RNASeq releasable", "notes": "v3-compatible"},



        {"column": "proteomics_meet_qc", "source": "multiomics Proteomics meet QC", "notes": "bool; blank→False"},



        {"column": "has_proteomics", "source": "multiomics Proteomics releasable", "notes": "v3-compatible"},



        {"column": "has_exposomics", "source": "multiomics Exposomics releasable v9", "notes": "bool; blank→False"},



        {



            "column": "has_srWGS",



            "source": "multiomics rid_no_srWGS",



            "notes": "True unless source marks person as lacking short-read WGS",



        },



        {"column": "age", "source": "part.csv.gz age_at_cdr", "notes": ""},



        {"column": "sex_at_birth", "source": "part.csv.gz (+ lr_1027_sex_at_birth.csv / merged_all_df.csv.gz fills)", "notes": "retains existing part value; supplied Phase 1 mapping then one-value-per-person merged_all fill missing usable values; sentinels are treated as missing in supplemental sources"},



        {"column": "zip3", "source": "part.csv.gz zip3_as_string", "notes": ""},



        {"column": "has_ehr_data", "source": "part.csv.gz (+ merged_all_df.csv.gz fill)", "notes": "nullable bool; merged_all fills only missing values when person-level value is unique"},



        {"column": "has_demographics", "source": "derived", "notes": "True if matched in part.csv.gz"},



        {"column": "extraction_method", "source": "lr_dna_extraction_method_cdrv9.tsv (+ HA sample fills)", "notes": "Chemagen or Autogen; CDR preferred, then ha_partial/ha_passing by biobank_id"},



        {"column": "has_extraction_method", "source": "derived", "notes": "True when extraction_method is non-missing (CDR or HA fill)"},



        {"column": "has_trgt", "source": "trgt_table.txt", "notes": "True if one or more TRGT VCFs are listed"},



        {"column": "n_trgt_vcfs", "source": "trgt_table.txt", "notes": "count of listed TRGT VCF paths"},



        {"column": "population", "source": "pc.log.gz member name", "notes": "population-specific PCA training set"},
        {"column": "ancestry_pred", "source": "anc_df.csv.gz (+ merged_all_df.csv.gz fill)", "notes": "external six-class ancestry prediction; merged_all fills missing values only when person-level value is unique"},
        {"column": "ancestry_pred_other", "source": "anc_df.csv.gz (+ merged_all_df.csv.gz fill)", "notes": "external ancestry prediction that permits oth; merged_all fills missing values only when person-level value is unique"},
        {"column": "has_ancestry_annotation", "source": "derived", "notes": "True if either ancestry prediction is non-missing after source supplements"},
        {"column": "in_pedigree", "source": "aou_phase2.ped", "notes": "True if person is represented in the user-maintained Phase 2 pedigree; Phase 1 has no pedigrees"},
        {"column": "pedigree_family_id", "source": "aou_phase2.ped ped", "notes": "family identifier; no parental IDs are exported"},
        {"column": "pedigree_is_founder", "source": "aou_phase2.ped father/mother", "notes": "True when both parent IDs are 0"},
        {"column": "pedigree_n_parents", "source": "aou_phase2.ped father/mother", "notes": "number of nonzero parents recorded (0–2)"},
        {"column": "pedigree_family_size", "source": "derived from aou_phase2.ped", "notes": "number of people in the recorded pedigree family"},



        *[



            {"column": f"PC{i}", "source": "full.tsv.gz global full-training PCA", "notes": "canonical PCs; match v3"}



            for i in range(1, 33)



        ],



        *[



            {"column": f"pop_PC{i}", "source": "pc.log.gz population-specific PCA", "notes": "within-population PCA axis"}



            for i in range(1, 33)



        ],



        {"column": "has_global_pcs", "source": "derived", "notes": "True if matched to a global full-training PCA row"},



        {"column": "has_pop_pcs", "source": "derived", "notes": "True if matched to a population-specific PCA row"},



        {"column": "in_cdr_v7", "source": "lr.tsv", "notes": "any repeated membership record marked v7"},



        {"column": "in_cdr_v8", "source": "lr.tsv", "notes": "any repeated membership record marked v8"},



        {"column": "in_cdr_v9", "source": "lr.tsv", "notes": "any repeated membership record marked v9"},



        {



            "column": "has_methylation",



            "source": "derived from lr_phase",



            "notes": "True for phase_2 and phase_1_phase_2; False for phase_1-only and unlabeled",



        },



        {



            "column": "in_phenotype_table",



            "source": "AoU_Phase2_Phenotype.csv.gz person_id",



            "notes": "True if research_id appears in the phenotype export; not trait ascertainment",



        },



        {"column": "biobank_id", "source": "final_set (+ ha_research_ids.csv fill)", "notes": "technical final-set value where available; HA mapping fills missing IDs"},



        {"column": "GC", "source": "final_set (+ Phase 1 fill)", "notes": "genome center; phase_1-only filled as HA (HudsonAlpha)"},



        {"column": "technology", "source": "final_set (+ Phase 1 fill)", "notes": "PacBio-prefer collapse when a v9 technical row exists; phase_1-only filled as PacBio"},



        {"column": "platform", "source": "final_set (+ Phase 1 fill)", "notes": "from collapsed technical row when available; phase_1-only filled as Sequel"},



        {"column": "pb_meth_caller", "source": "final_set PacBioMethylationCaller (+ Phase 1 fill)", "notes": "phase_1-only filled as none (no methylation)"},



        {"column": "inferred_sex", "source": "final_set sex", "notes": ""},



        {"column": "coverage", "source": "final_set / Integratedcall-GRCh38.tsv (+ merged_all_df.csv.gz fill)", "notes": "v9 technical coverage where available; Phase 1-only uses integratedcall aligned fold coverage; merged_all fills only missing values when person-level value is unique"},
        {"column": "coverage_source", "source": "derived", "notes": "v9_technical, integratedcall_phase1, or merged_all_df; NA when coverage is unavailable"},
        {"column": "ont_coverage", "source": "ont-sample-hg38.tsv mosdepth_cov", "notes": "separate ONT mosdepth coverage for curated Phase 1 ONT samples; never substitutes for coverage"},
        {"column": "ont_aligned_num_bases", "source": "ont-sample-hg38.tsv aligned_num_bases", "notes": "curated Phase 1 ONT aligned bases"},
        {"column": "ont_aligned_num_reads", "source": "ont-sample-hg38.tsv aligned_num_reads", "notes": "curated Phase 1 ONT aligned reads"},
        {"column": "ont_read_length_n50", "source": "ont-sample-hg38.tsv aligned_read_length_N50", "notes": "curated Phase 1 ONT read-length N50"},
        {"column": "ont_read_length_mean", "source": "ont-sample-hg38.tsv aligned_read_length_mean", "notes": "curated Phase 1 ONT mean read length"},
        {"column": "ont_read_length_median", "source": "ont-sample-hg38.tsv aligned_read_length_median", "notes": "curated Phase 1 ONT median read length"},
        {"column": "ont_read_length_stdev", "source": "ont-sample-hg38.tsv aligned_read_length_stdev", "notes": "curated Phase 1 ONT read-length standard deviation"},
        {"column": "ont_average_identity", "source": "ont-sample-hg38.tsv average_identity", "notes": "curated Phase 1 ONT average alignment identity (percent)"},
        {"column": "ont_median_identity", "source": "ont-sample-hg38.tsv median_identity", "notes": "curated Phase 1 ONT median alignment identity (percent)"},
        {
            "column": "has_sv_counts",
            "source": "counts.tar.gz",
            "notes": "True if a Phase 2 caller-specific SV-count record is available after center-matched duplicate resolution",
        },
        {
            "column": "n_sv_del",
            "source": "derived from counts.tar.gz / sv_sens_07_counts.json",
            "notes": "Phase 2: max deletion count across PAV/PBSV/Sniffles ≥50 bp; Phase 1: sens07 DEL; Phase 2 value takes precedence for re-sequenced people",
        },
        {
            "column": "n_sv_ins",
            "source": "derived from counts.tar.gz / sv_sens_07_counts.json",
            "notes": "Phase 2: max insertion count across PAV/PBSV/Sniffles ≥50 bp; Phase 1: sens07 INS; Phase 2 value takes precedence for re-sequenced people",
        },



        {"column": "is_AIAN", "source": "final_set", "notes": ""},



        {



            "column": "withdrawn",



            "source": "final_set withdrew",



            "notes": "any TRUE across final-set rows; authoritative consent withdrawal",



        },



        {



            "column": "unavailable",



            "source": "derived from final-set",



            "notes": "in final-set, not withdrawn, and no v9-releasable row (set-aside)",



        },



        {



            "column": "final_releasable_v9",



            "source": "final_set long_read releasable flag",



            "notes": "any releasable=1 across final-set rows",



        },



        {



            "column": "in_final_set",



            "source": "derived",



            "notes": "True if research_id appears in Steve final-set",



        },



        {



            "column": "n_final_rows",



            "source": "final_set",



            "notes": "number of final-set sequencing rows for this ID",



        },



        {"column": "has_lr_tech", "source": "derived", "notes": "in ONT∪PacBio v9 sheets"},



        {"column": "has_ONT", "source": "ONT_v9 (+ Phase 1 curated ONT list)", "notes": "True if in CDRv9 ONT sheet or curated Phase 1 additional-ONT research IDs"},



        {"column": "has_PacBio", "source": "PacBio_v9", "notes": "True if present in CDRv9 PacBio lrWGS sheet"},



        {"column": "n_lr_technical_rows", "source": "final_set (+ Phase 1 fill)", "notes": "number of long-read technical source rows before person-level collapse; Phase 1-only set to 1 for its known single PacBio run"},



        {"column": "techs_available", "source": "final_set (+ Phase 1 fill / Phase 1 ONT list)", "notes": "technologies observed before collapse; phase_1-only default PacBio; curated Phase 1 ONT list → ONT|PacBio"},



    ]



)







assert list(data_dictionary["column"]) == out_cols







out.to_csv(OUT_CSV, index=False, compression="gzip")



data_dictionary.to_csv(OUT_DICT, sep="\t", index=False)



print(f"wrote {OUT_CSV} ({len(out):,} rows, {out.shape[1]} cols)")



print(f"wrote {OUT_DICT}")



display(out.head())



display(out[["age", "sex_at_birth", "has_ehr_data", "coverage", "has_rna"]].isna().mean().rename("frac_missing"))



print("construction complete; v3 was not used:", not CONSTRUCTION_USED_V3)





## 5. Compare to `covariates.v3.csv.gz` (read-only)

v3 is loaded **only after** the source-built file exists.


In [ ]:
assert OUT_CSV.exists()



assert V3_CSV.exists()



COMPARE_LOADED_V3 = True







rebuilt = pd.read_csv(OUT_CSV, dtype={"research_id": str, "biobank_id": str})



v3 = pd.read_csv(V3_CSV, dtype={"research_id": str, "biobank_id": str})







print(f"rebuilt: {len(rebuilt):,} IDs")



print(f"v3:      {len(v3):,} rows, {v3['research_id'].nunique():,} unique")







rebuilt_ids = set(rebuilt["research_id"])



v3_ids = set(v3["research_id"])



overlap = rebuilt_ids & v3_ids



print(f"ID overlap: {len(overlap):,}; only rebuilt: {len(rebuilt_ids - v3_ids):,}; only v3: {len(v3_ids - rebuilt_ids):,}")







# Align renamed rebuilt columns to v3 names for field-level comparison only.



REBUILT_TO_V3 = {



    "zip3": "zip3_as_string",



    "has_trgt": "has_trgt_calls",



    "pb_meth_caller": "PacBioMethylationCaller",



}



rebuilt_for_cmp = rebuilt.rename(columns=REBUILT_TO_V3)







COMMON_COMPARE_COLS = [



    "biobank_id",



    "GC",



    "technology",



    "platform",



    "PacBioMethylationCaller",



    "has_rna",



    "has_proteomics",



    "inferred_sex",



    "coverage",



    "is_AIAN",



    "age",



    "sex_at_birth",



    "zip3_as_string",



    "has_ehr_data",



    "extraction_method",



    "has_trgt_calls",



    "population",



    "in_cdr_v7",



    "in_cdr_v8",



    "in_cdr_v9",



    *PC_COLS,



]







print("\nv3-only columns:", sorted(set(v3.columns) - set(rebuilt_for_cmp.columns)))



print("rebuilt-only columns:", sorted(set(rebuilt_for_cmp.columns) - set(v3.columns)))



print("PC1–PC32 use full.tsv.gz global full-training PCA; pop_PC1–pop_PC32 retain the population-specific coordinates.")







v3_work = v3.copy()



v3_work["coverage_num"] = pd.to_numeric(v3_work["coverage"], errors="coerce")



v3_work["tech_rank"] = np.where(v3_work["technology"] == "PacBio", 0, 1)



v3_work["platform_key"] = v3_work["platform"].fillna("").astype(str)



v3_work["meth_key"] = v3_work["PacBioMethylationCaller"].fillna("").astype(str)



v3_work["biobank_key"] = v3_work["biobank_id"].fillna("").astype(str)



v3_work = v3_work.sort_values(



    by=["research_id", "tech_rank", "coverage_num", "platform_key", "meth_key", "biobank_key"],



    ascending=[True, True, False, True, True, True],



    kind="mergesort",



)



v3_person = v3_work.groupby("research_id", sort=False, as_index=False).head(1).copy()







cmp = rebuilt_for_cmp.loc[rebuilt_for_cmp["research_id"].isin(overlap), ["research_id"] + COMMON_COMPARE_COLS].merge(



    v3_person.loc[v3_person["research_id"].isin(overlap), ["research_id"] + COMMON_COMPARE_COLS],



    on="research_id",



    how="inner",



    suffixes=("_rebuilt", "_v3"),



    validate="one_to_one",



)











def series_equal(a: pd.Series, b: pd.Series) -> pd.Series:



    a_num = pd.to_numeric(a, errors="coerce")



    b_num = pd.to_numeric(b, errors="coerce")



    both_numeric = a_num.notna() & b_num.notna()



    num_eq = np.isclose(a_num.to_numpy(dtype=float), b_num.to_numpy(dtype=float), equal_nan=False)



    a_str = a.fillna("").astype(str).str.strip()



    b_str = b.fillna("").astype(str).str.strip()



    bool_map = {



        "True": "True", "TRUE": "True", "true": "True", "1": "True",



        "False": "False", "FALSE": "False", "false": "False", "0": "False",



        "": "", "nan": "", "<NA>": "", "None": "",



    }



    a_str = a_str.map(lambda x: bool_map.get(x, x))



    b_str = b_str.map(lambda x: bool_map.get(x, x))



    both_null = a.isna() & b.isna()



    return pd.Series(np.where(both_numeric, num_eq, a_str.eq(b_str) | both_null.to_numpy()), index=a.index)











agree_rows = []



mismatch_frames = {}



for col in COMMON_COMPARE_COLS:



    eq = series_equal(cmp[f"{col}_rebuilt"], cmp[f"{col}_v3"])



    agree_rows.append({"column": col, "n_agree": int(eq.sum()), "n_disagree": int((~eq).sum()), "frac_agree": eq.mean()})



    if (~eq).any():



        mismatch_frames[col] = cmp.loc[~eq, ["research_id", f"{col}_rebuilt", f"{col}_v3"]]







display(pd.DataFrame(agree_rows))



all_eq = pd.Series(True, index=cmp.index)



for col in COMMON_COMPARE_COLS:



    all_eq &= series_equal(cmp[f"{col}_rebuilt"], cmp[f"{col}_v3"])



print(f"full-row agreement: {int(all_eq.sum()):,} / {len(cmp):,}")



if mismatch_frames:



    for col, bad in mismatch_frames.items():



        print(f"\n### {col} ({len(bad)} disagree)")



        display(bad.head(5))



print("Output:", OUT_CSV)



